<a href="https://colab.research.google.com/github/tsal4/data2000_labs/blob/main/homework/040_linear-models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [142]:
!pip install ucimlrepo

# Homework Assignment: Predicting Heart Disease with Logistic Regression

## Overview

In this lab you will work with a real clinical dataset collected from patients undergoing cardiac evaluation at four medical institutions in the 1980s. Your goal is to build, evaluate, and interpret a binary logistic regression model that predicts whether a patient has heart disease based on a set of diagnostic measurements. Along the way you will practice the full modeling workflow: loading and inspecting raw data, cleaning and encoding features, fitting a logistic regression, and interpreting your results in clinically meaningful terms.

---

## The Dataset

The UCI Heart Disease dataset is one of the most widely studied datasets in machine learning, and for good reason — it is small enough to be tractable, rich enough to be interesting, and grounded in real clinical practice. The data were originally collected across four sites: the Cleveland Clinic Foundation, the Hungarian Institute of Cardiology in Budapest, the University Hospital in Zurich, and the University Hospital in Long Beach. The version most commonly used in practice, and the one you will work with here, comes from Cleveland and contains **303 patient records**.

Each row represents a single patient. The original dataset contains a target variable (`num`) ranging from 0 to 4, indicating the degree of coronary artery narrowing observed during angiography. For this assignment you will **binarize the target**: patients with a value of 0 will be labeled **0 (no heart disease)** and patients with values 1 through 4 will be labeled **1 (heart disease present)**. This reflects the clinically relevant question: does this patient have meaningful coronary artery disease, or not?

---

## Features

The dataset contains 13 predictor variables. Read these descriptions carefully — understanding what each variable actually measures will help you make better decisions during cleaning and will make your interpretation more meaningful.

**`age`** — Age of the patient in years. A continuous variable. Older age is generally associated with higher cardiovascular risk, though the relationship is not perfectly linear.

**`sex`** — Biological sex, encoded as 1 (male) and 0 (female). In the original study population, male patients had higher rates of disease. You should treat this as a binary categorical feature.

**`cp`** — Chest pain type. This is a categorical variable with four values: 1 = typical angina, 2 = atypical angina, 3 = non-anginal pain, 4 = asymptomatic. Typical angina is the classic presentation of cardiac chest pain — pressure or tightness brought on by exertion and relieved by rest. Importantly, asymptomatic patients (value 4) actually show *higher* rates of disease in this dataset, which is a good reminder not to assume ordinal relationships in categorical variables. You should one-hot encode this feature.

**`trestbps`** — Resting blood pressure in mmHg, measured at the time of hospital admission. Normal resting blood pressure is around 120/80 mmHg; elevated values may indicate hypertension and increased cardiac strain. This is a continuous feature and may contain a small number of physiologically implausible outliers worth inspecting.

**`chol`** — Serum cholesterol in mg/dL. Higher cholesterol, particularly LDL cholesterol, is a well-established risk factor for coronary artery disease. This is continuous. Be aware that a small number of records contain a value of 0, which is physiologically impossible and should be treated as missing.

**`fbs`** — Fasting blood sugar, recorded as a binary indicator: 1 if fasting blood sugar is greater than 120 mg/dL, 0 otherwise. Elevated fasting blood sugar is a marker of diabetes or pre-diabetes, both of which significantly increase cardiovascular risk. Treat this as binary categorical.

**`restecg`** — Resting electrocardiographic results. Another categorical variable with three values: 0 = normal, 1 = ST-T wave abnormality (which can indicate ischemia or other cardiac stress), 2 = left ventricular hypertrophy by Estes' criteria. You should one-hot encode this feature, though be aware that value 2 is rare in this dataset and its coefficient may be unstable.

**`thalach`** — Maximum heart rate achieved during exercise stress testing. This is the peak heart rate recorded during a supervised treadmill or cycling test. Higher maximum heart rate is generally associated with better cardiovascular fitness and lower disease risk, so you may find this feature has a negative relationship with the outcome. This is a continuous variable.

**`exang`** — Exercise-induced angina, encoded as 1 (yes) or 0 (no). This indicates whether the patient experienced chest pain during the exercise stress test. It is one of the most clinically informative features in the dataset, as angina triggered by exertion is a strong indicator of obstructed coronary blood flow. Treat this as binary categorical.

**`oldpeak`** — ST depression induced by exercise relative to rest, measured in millivolts. During an ECG stress test, depression of the ST segment is a sign that part of the heart muscle is not receiving adequate blood flow. A value of 0 means no depression was observed; higher values indicate greater depression and more severe ischemia. This is a continuous feature with a right-skewed distribution and a notable mass of values at exactly 0.

**`slope`** — The slope of the peak exercise ST segment. This describes the shape of the ST segment at maximum exercise: 1 = upsloping, 2 = flat, 3 = downsloping. A downsloping or flat ST segment during peak exercise is considered more clinically concerning than an upsloping one. This should be treated as categorical and one-hot encoded.

**`ca`** — Number of major coronary vessels colored by fluoroscopy, ranging from 0 to 3. Fluoroscopy is an imaging technique used to visualize blood flow through the coronary arteries; a vessel that "colors" well is unobstructed. More obstructed vessels indicates more widespread disease. This is recorded as a numeric variable but functions more like an ordinal count. **It contains missing values coded as `?` in the raw file**, which you will need to handle.

**`thal`** — Results of a thallium stress test, a nuclear imaging procedure that reveals areas of the heart with reduced blood flow. Values are: 3 = normal, 6 = fixed defect (an area that never receives adequate blood flow, often indicating prior heart attack), 7 = reversible defect (an area with reduced flow during stress that recovers at rest, indicating ischemia). This is categorical and should be one-hot encoded. Like `ca`, **it contains missing values coded as `?`** in the raw file.

---

## Your Tasks

1. **Load and inspect** the raw data. Assign column names, identify data types, and produce a summary of missing values and basic descriptive statistics for each feature.

2. **Clean the data.** Address the missing value issues described above. Justify your imputation strategy — mean, median, mode, or drop — for each affected feature, with a brief explanation of why that choice makes sense given the variable's distribution and clinical meaning.

3. **Engineer your features.** Binarize the target variable. One-hot encode the appropriate categorical features. Decide how to handle the binary categoricals and document your reasoning.

4. **Fit a logistic regression model** using a train-test split. Report accuracy, a confusion matrix, and other relevant metrics for interpreting model performance on the test set. Given the clinical context, reflect briefly on whether accuracy is the right metric here, or whether another metric deserves more weight.

In [6]:
import pprint
from ucimlrepo import fetch_ucirepo
import gdown
import pandas as pd
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
from sklearn.metrics import PredictionErrorDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import RocCurveDisplay

In [7]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [8]:
heart_disease = fetch_ucirepo(id=45)
X = heart_disease.data.features
y = heart_disease.data.targets

In [9]:
#pprint.pprint(heart_disease.metadata, indent=4)
#pprint.pprint(heart_disease.variables, indent=4)

In [10]:
X.head(12)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0
5,56,1,2,120,236,0,0,178,0,0.8,1,0.0,3.0
6,62,0,4,140,268,0,2,160,0,3.6,3,2.0,3.0
7,57,0,4,120,354,0,0,163,1,0.6,1,0.0,3.0
8,63,1,4,130,254,0,2,147,0,1.4,2,1.0,7.0
9,53,1,4,140,203,1,2,155,1,3.1,3,0.0,7.0


In [11]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 303 entries, 0 to 302
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       303 non-null    int64  
 1   sex       303 non-null    int64  
 2   cp        303 non-null    int64  
 3   trestbps  303 non-null    int64  
 4   chol      303 non-null    int64  
 5   fbs       303 non-null    int64  
 6   restecg   303 non-null    int64  
 7   thalach   303 non-null    int64  
 8   exang     303 non-null    int64  
 9   oldpeak   303 non-null    float64
 10  slope     303 non-null    int64  
 11  ca        299 non-null    float64
 12  thal      301 non-null    float64
dtypes: float64(3), int64(10)
memory usage: 30.9 KB


In [12]:
X.describe()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
count,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,303.000000,299.000000,301.000000
mean,54.438944,0.679868,3.158416,131.689769,246.693069,0.148515,0.990099,149.607261,0.326733,1.039604,1.600660,0.672241,4.734219
std,9.038662,0.467299,0.960126,17.599748,51.776918,0.356198,0.994971,22.875003,0.469794,1.161075,0.616226,0.937438,1.939706
min,29.000000,0.000000,1.000000,94.000000,126.000000,0.000000,0.000000,71.000000,0.000000,0.000000,1.000000,0.000000,3.000000
25%,48.000000,0.000000,3.000000,120.000000,211.000000,0.000000,0.000000,133.500000,0.000000,0.000000,1.000000,0.000000,3.000000
50%,56.000000,1.000000,3.000000,130.000000,241.000000,0.000000,1.000000,153.000000,0.000000,0.800000,2.000000,0.000000,3.000000
75%,61.000000,1.000000,4.000000,140.000000,275.000000,0.000000,2.000000,166.000000,1.000000,1.600000,2.000000,1.000000,7.000000
max,77.000000,1.000000,4.000000,200.000000,564.000000,1.000000,2.000000,202.000000,1.000000,6.200000,3.000000,3.000000,7.000000


In [13]:
# List of columns to clean: ***cp(one-hot encode), ***trestbps(outliers), ***chol(impossible to have 0), ***restecg(one-hot encode), ***slope(one-hot encode),
# ***ca(values of nan), ***thal(values of nan)

# num (binarize, 0 => 0, 1-4 => 0)

In [14]:
X.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63,1,1,145,233,1,2,150,0,2.3,3,0.0,6.0
1,67,1,4,160,286,0,2,108,1,1.5,2,3.0,3.0
2,67,1,4,120,229,0,2,129,1,2.6,2,2.0,7.0
3,37,1,3,130,250,0,0,187,0,3.5,3,0.0,3.0
4,41,0,2,130,204,0,2,172,0,1.4,1,0.0,3.0


In [15]:
#One-hot encoding
cols_to_encode = ['cp', 'restecg', 'slope']
X = pd.get_dummies(X, columns=cols_to_encode, prefix=cols_to_encode, prefix_sep='-')

#X = pd.concat([X, X_ohe], axis=1)

In [16]:
X.head()

,age,sex,trestbps,chol,fbs,thalach,exang,oldpeak,ca,thal,cp-1,cp-2,cp-3,cp-4,restecg-0,restecg-1,restecg-2,slope-1,slope-2,slope-3
0,63,1,145,233,1,150,0,2.3,0.0,6.0,True,False,False,False,False,False,True,False,False,True
1,67,1,160,286,0,108,1,1.5,3.0,3.0,False,False,False,True,False,False,True,False,True,False
2,67,1,120,229,0,129,1,2.6,2.0,7.0,False,False,False,True,False,False,True,False,True,False
3,37,1,130,250,0,187,0,3.5,0.0,3.0,False,False,True,False,True,False,False,False,False,True
4,41,0,130,204,0,172,0,1.4,0.0,3.0,False,True,False,False,False,False,True,True,False,False


In [17]:
#X["trestbps"].plot.hist()

In [18]:
#trestbps outliers
for val in (0, 0.01, 0.1, 0.25, 0.75, 0.9, 0.95, 0.98, 0.99, 0.995, 0.999, 0.9999, 1):
    print(f"{val}:\t {X['trestbps'].quantile(val)}")

0:	 94.0
0.01:	 100.0
0.1:	 110.0
0.25:	 120.0
0.75:	 140.0
0.9:	 152.0
0.95:	 160.0
0.98:	 177.83999999999992
0.99:	 180.0
0.995:	 185.8800000000001
0.999:	 197.5840000000003
0.9999:	 199.7583999999997
1:	 200.0


After some research it seemed like 95 is the lowest blood pressure can really get before it's fatal, so I will make that the low boundary. Anything above 180 is a medical emergency, and probably would not be someone's resting blood pressure. I will make 180 the high boundary. I set these high to keep as much data as possible.

In [19]:
X = X.loc[X["trestbps"] <= 180, :]

In [20]:
X = X.loc[X["trestbps"] >= 95, :]

In [21]:
X.describe()

,age,sex,trestbps,chol,fbs,thalach,exang,oldpeak,ca,thal
count,299.000000,299.000000,299.000000,299.000000,299.000000,299.000000,299.000000,299.000000,295.000000,297.000000
mean,54.498328,0.682274,131.511706,246.658863,0.147157,149.397993,0.324415,1.040134,0.667797,4.717172
std,9.052188,0.466373,16.627654,51.939860,0.354856,22.791265,0.468941,1.151478,0.939395,1.936703
min,29.000000,0.000000,100.000000,126.000000,0.000000,71.000000,0.000000,0.000000,0.000000,3.000000
25%,48.000000,0.000000,120.000000,211.000000,0.000000,133.500000,0.000000,0.000000,0.000000,3.000000
50%,56.000000,1.000000,130.000000,241.000000,0.000000,152.000000,0.000000,0.800000,0.000000,3.000000
75%,61.000000,1.000000,140.000000,274.500000,0.000000,165.500000,1.000000,1.600000,1.000000,7.000000
max,77.000000,1.000000,180.000000,564.000000,1.000000,202.000000,1.000000,6.200000,3.000000,7.000000


In [22]:
for val in (0, 0.01, 0.1, 0.25, 0.75, 0.9, 0.95, 0.98, 0.99, 0.995, 0.999, 0.9999, 1):
    print(f"{val}:\t {X['chol'].quantile(val)}")

0:	 126.0
0.01:	 148.84
0.1:	 188.0
0.25:	 211.0
0.75:	 274.5
0.9:	 309.0
0.95:	 327.2999999999999
0.98:	 354.2400000000001
0.99:	 407.03999999999996
0.995:	 413.0799999999999
0.999:	 520.1940000000081
0.9999:	 559.6193999999966
1:	 564.0


Removing outliers for trestbps also seemed to remove the cholesterol outliers.

In [23]:
X.info()

<class 'pandas.core.frame.DataFrame'>
Index: 299 entries, 0 to 302
Data columns (total 20 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   age        299 non-null    int64  
 1   sex        299 non-null    int64  
 2   trestbps   299 non-null    int64  
 3   chol       299 non-null    int64  
 4   fbs        299 non-null    int64  
 5   thalach    299 non-null    int64  
 6   exang      299 non-null    int64  
 7   oldpeak    299 non-null    float64
 8   ca         295 non-null    float64
 9   thal       297 non-null    float64
 10  cp-1       299 non-null    bool   
 11  cp-2       299 non-null    bool   
 12  cp-3       299 non-null    bool   
 13  cp-4       299 non-null    bool   
 14  restecg-0  299 non-null    bool   
 15  restecg-1  299 non-null    bool   
 16  restecg-2  299 non-null    bool   
 17  slope-1    299 non-null    bool   
 18  slope-2    299 non-null    bool   
 19  slope-3    299 non-null    bool   
dtypes: bool(10), fl

Now I will be filling the nan of "ca" and "thal" with their modes. I chose the mode because these are categorical variables. The mode is the most common value, so I figured that if you had to guess, these rows maybe probably have the most common value as well.

In [24]:
X['ca'] = X['ca'].fillna(X['ca'].mode()[0])

In [25]:
X['thal'] = X['thal'].fillna(X['thal'].mode()[0])

In [26]:
X.info()

<class 'pandas.core.frame.DataFrame'>
Index: 299 entries, 0 to 302
Data columns (total 20 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   age        299 non-null    int64  
 1   sex        299 non-null    int64  
 2   trestbps   299 non-null    int64  
 3   chol       299 non-null    int64  
 4   fbs        299 non-null    int64  
 5   thalach    299 non-null    int64  
 6   exang      299 non-null    int64  
 7   oldpeak    299 non-null    float64
 8   ca         299 non-null    float64
 9   thal       299 non-null    float64
 10  cp-1       299 non-null    bool   
 11  cp-2       299 non-null    bool   
 12  cp-3       299 non-null    bool   
 13  cp-4       299 non-null    bool   
 14  restecg-0  299 non-null    bool   
 15  restecg-1  299 non-null    bool   
 16  restecg-2  299 non-null    bool   
 17  slope-1    299 non-null    bool   
 18  slope-2    299 non-null    bool   
 19  slope-3    299 non-null    bool   
dtypes: bool(10), fl

In [27]:
y.head(12)

,num
0,0
1,2
2,1
3,0
4,0
5,0
6,3
7,0
8,2
9,1


In [28]:
#Binarize y
y['num'] = y['num'].apply(
    lambda x: 0 if x == 0 else 1
)

/tmp/ipython-input-13801/2640253816.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y['num'] = y['num'].apply(


In [29]:
y.head(12)

,num
0,0
1,1
2,1
3,0
4,0
5,0
6,1
7,0
8,1
9,1


## Data cleaning is DONE, onto modeling